In [1]:
from tensorflow.keras.models import load_model
import cv2
import mediapipe as mp
import numpy as np

model = load_model("yoga_pose_model.h5")



pose_labels = ["Downdog", "Goddess", "Plank", "Tree", "Warrior2"]


/Users/gurparkashsingh/Desktop/Machine Learning/Image_Pose_Recognition/.venv/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/gurparkashsingh/Desktop/Machine Learning/Image_Pose_Recognition/.venv/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/gurparkashsingh/Desktop/Machine Learning/Image_Pose_Recognition/.venv/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
print(model.input_shape)


(None, 99)


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf

# Load your trained model
model = tf.keras.models.load_model("yoga_pose_model.h5")
pose_labels = ["Downdog", "Goddess", "Plank", "Tree", "Warrior2"]

# MediaPipe setup
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)
mp_drawing = mp.solutions.drawing_utils

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

# --- Load your image here ---
image_path = "YogaPoses/Plank/00000005.jpg"  # Replace with your image file
image = cv2.imread(image_path)
if image is None:
    raise FileNotFoundError(f"Image not found at {image_path}")
h, w, _ = image.shape
# Convert BGR to RGB
rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Process pose
results = pose.process(rgb_image)

pose_class = "Not Detected"
feedback = ""

if results.pose_landmarks:
    landmarks = results.pose_landmarks.landmark

    # Bounding box check
    x_coords = [lm.x for lm in landmarks]
    y_coords = [lm.y for lm in landmarks]
    bbox_width = max(x_coords) - min(x_coords)
    bbox_height = max(y_coords) - min(y_coords)
    too_close = bbox_width > 0.7 or bbox_height > 0.7

    required = [11, 12, 23, 24, 27, 28]
    visible = all(landmarks[i].visibility > 0.5 for i in required)

    if not too_close and visible:
        keypoints = []
        for lm in landmarks:
            keypoints.extend([lm.x, lm.y, lm.z])
        input_data = np.array(keypoints).reshape(1, -1)

        prediction = model.predict(input_data, verbose=0)
        class_id = np.argmax(prediction)
        confidence = np.max(prediction)

        if confidence > 0.7:
            pose_class = f"{pose_labels[class_id]} ({confidence:.2f})"
        else:
            pose_class = "Uncertain"

        # If plank, give feedback
        if "Plank" in pose_class:
            l_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x * w,
                          landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y * h]
            l_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x * w,
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y * h]
            l_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x * w,
                       landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y * h]

            body_angle = calculate_angle(l_shoulder, l_hip, l_ankle)
            print("feedback is")
            if body_angle < 160:
                feedback = "Hips too low! Lift up."
            elif body_angle > 200:
                feedback = "Hips too high! Lower down."
            else:
                feedback = "Good plank!"

            cv2.line(image, tuple(np.array(l_shoulder, dtype=int)),
                     tuple(np.array(l_hip, dtype=int)), (0, 255, 255), 3)
            cv2.line(image, tuple(np.array(l_hip, dtype=int)),
                     tuple(np.array(l_ankle, dtype=int)), (0, 255, 255), 3)
            if feedback:
                print("Feedback:", feedback)

    else:
        pose_class = "Pose not too Close" if too_close else "Not Visible"

    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

# Add text to image
cv2.putText(image, f"Pose: {pose_class}", (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

if feedback:
    cv2.putText(image, feedback, (50, h - 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

# Show the result
cv2.imshow("Yoga Pose Result", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

I0000 00:00:1756932992.934119 2953800 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1756932992.994545 2968654 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1756932993.005886 2968663 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


: 